# 365 Probabilidades - Dia #155
## Qual a probabilidade de o seu cachorro saber que você está estressado antes de você dizer?

**Tipo:** Curiosidade · Animais
**Data de publicação:** 2026-11-15
**Ferramenta:** Python
**Decisão analisada:** Quando o cachorro muda de jeito num dia ruim, é coincidência ou ele percebeu?
**Hashtag:** #365Probabilidades #Dia155

---

### 📖 A História

Tem dias bem mais difíceis do que outros. Mais tensos, mais estressantes. Trabalho quase sempre de casa, e as minhas cachorras costumam ficar por perto. Num desses dias, me veio a pergunta de hoje. Será que os cachorros sentem quando a gente está assim?

Quem tem cachorro costuma jurar que ele sabe quando o dia foi ruim. Ele chega mais perto, fica quieto, ou fica agitado sem motivo aparente. Fui ver se isso se mede, ou se é só uma coisa que a gente gosta de acreditar.

---

### 📚 O Conceito: o estresse tem cheiro

Quando uma pessoa fica estressada, o corpo muda depressa. O coração acelera, a pressão sobe,
o suor e a respiração mudam de composição. Parte dessas mudanças sai pelo ar, em moléculas
que o nariz humano não percebe. O nariz de um cachorro tem dezenas de vezes mais receptores
de cheiro que o nosso.

Se o estresse muda o cheiro, o cachorro teria como saber antes de qualquer palavra ou gesto.
Para testar isso de verdade, é preciso tirar a pessoa da sala e deixar só o cheiro.

---

### 🧮 O Modelo

**Fontes:**
- Wilson, C., Campbell, K., Petzel, Z. & Reeve, C. (2022). Dogs can discriminate between human
  baseline and psychological stress condition odours. *PLOS ONE*, 17(9), e0274143. Queen's
  University Belfast. 36 pessoas deram amostras de hálito e suor antes e depois de uma tarefa
  de contas de cabeça sob pressão; o estresse foi confirmado por autorrelato (escala de 0 a 10)
  e, em 26 delas, por frequência cardíaca e pressão. 4 cachorros treinados, teste duplo-cego,
  3 potes (a amostra estressada, a amostra calma da mesma pessoa e um pote vazio), 20
  tentativas por pessoa. **675 acertos em 720 tentativas.** Dados por sessão do material
  suplementar (S1 Dataset), baixados por Ana em 08/11/2026 e organizados em
  `dados155/dia-155-wilson2022-sessoes.csv`. Nota de autoria: Clara Wilson não é o Timothy D.
  Wilson do #029.
- D'Aniello, B., Semin, G. R., Alterisio, A., Aria, M. & Scandurra, A. (2018).
  Interspecies transmission of emotional information via chemosignals: from humans to dogs.
  *Animal Cognition*, 21(1), 67-78. Labradores e golden retrievers expostos ao suor de pessoas
  com medo, felizes ou a nenhum suor: com o cheiro de medo, mais sinais de estresse e
  frequência cardíaca mais alta. Lido pelo resumo, fora do modelo.
- Parr-Cortes, Z. et al. (2024). The odour of an unfamiliar stressed or relaxed person affects
  dogs' responses to a cognitive bias test. *Scientific Reports*, 14. Universidade de Bristol.
  18 cachorros comuns: com o cheiro de uma pessoa estressada, mesmo desconhecida, se
  aproximaram menos de um pote em posição ambígua, sinal de expectativa mais pessimista. Lido
  pelo resumo, fora do modelo.
- Sundman, A.-S. et al. (2019). Long-term stress levels are synchronized in dogs and their
  owners. *Scientific Reports*, 9, 7391. Universidade de Linköping. 58 duplas (33 shetland
  sheepdogs e 25 border collies, todas as donas mulheres). Cortisol no pelo e no cabelo,
  medido no verão e no inverno. O cortisol da dona acompanhou o do cachorro nas duas
  estações (modelos lineares generalizados, verão χ² = 23,7, p < 0,001, N = 57; inverno
  χ² = 13,8, p < 0,001, N = 55), mais forte nos cachorros de competição e nas fêmeas.
  Associação, não causa. O artigo tem uma correção publicada em 2020 que não foi lida.

**Desenho e família do teste.** Escolha forçada entre três potes, com acerto ao acaso de 1 em
3. Cada sessão é uma pessoa, com 20 tentativas.
- Posterior Beta de Jeffreys para a taxa de acerto, no total e por cachorro.
- Teste binomial de cada pessoa contra o acaso.
- Modelo beta-binomial ajustado às 36 sessões, para separar a variação de pessoa para pessoa
  do acaso dentro da sessão, e Monte Carlo de 200 mil pessoas novas.
- Correlação de Spearman entre o tamanho do estresse e o acerto.

**Sem ×0,80.** São acertos contados num teste duplo-cego e medidas de cortisol.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize, special

SEMENTE = 155
rng = np.random.default_rng(SEMENTE)
SORTEIOS = 200_000
PASTA = 'dados155'
ACASO = 1 / 3
N_TENT = 20

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = ('Fonte: Wilson, Campbell, Petzel & Reeve 2022, PLOS ONE 17(9), dados S1 '
          '· cálculo próprio | #365Probabilidades')

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def jeffreys(k, n):
    return stats.beta(k + 0.5, n - k + 0.5)

print("✅ Bibliotecas carregadas · semente", SEMENTE)


In [ ]:
# --- DADOS ---
ARQ = os.path.join(PASTA, 'dia-155-wilson2022-sessoes.csv')
if not os.path.exists(ARQ):
    bruto = pd.read_excel(os.path.join(PASTA, 'wilson2022-s1.xlsx'), 'Tested Samples')
    col = bruto.columns
    pd.DataFrame({
        'participante': bruto['Participant Number'], 'remoto': bruto['Remote or In-Person'],
        'genero': bruto['Gender'], 'idade': bruto['Age (range provided for anonymity)'],
        'vas_base': bruto['VAS Baseline'], 'vas_mat': bruto['VAS MAT'],
        'fc_base': bruto['Baseline HR'], 'fc_mat': bruto['MAT HR'],
        'pa_base': bruto['Baseline BP'], 'pa_mat': bruto['MAT BP'],
        'cachorro': bruto['Dog Tested'].astype(str).str.strip(),
        'fase1_acertos': bruto[col[18]], 'acertos': bruto[col[19]], 'tentativas': N_TENT,
    }).to_csv(ARQ, index=False)

d = pd.read_csv(ARQ)
d['vas_dif'] = d['vas_mat'] - d['vas_base']
d['fc_dif'] = d['fc_mat'] - d['fc_base']
K, N = int(d['acertos'].sum()), int(d['tentativas'].sum())

print("=" * 72)
print("  DADOS · WILSON ET AL. 2022")
print("=" * 72)
print(f"  Pessoas testadas: {len(d)} · tentativas: {N} · acertos: {K} ({vir(K / N * 100, 2)}%)")
print(f"  Estresse relatado (0 a 10): de {vir(d['vas_base'].mean(), 2)} para {vir(d['vas_mat'].mean(), 2)}")
fc = d.dropna(subset=['fc_base'])
print(f"  Frequência cardíaca ({len(fc)} pessoas): de {vir(fc['fc_base'].mean())} para {vir(fc['fc_mat'].mean())} bpm")
print("\n  Por cachorro:")
print(d.groupby('cachorro').agg(pessoas=('acertos', 'size'), acertos=('acertos', 'sum')).to_string())
print("=" * 72)


In [ ]:
# --- O MODELO ---

# CAMADA 1 · O acerto, no total e por cachorro
print("=" * 72)
print("  CAMADA 1 · O ACERTO (posterior de Jeffreys)")
print("=" * 72)
TOTAL = jeffreys(K, N)
lo, hi = TOTAL.interval(0.95)
print(f"  Todos os cachorros: {vir(K / N * 100)}%  IC 95% [{vir(lo * 100)}; {vir(hi * 100)}]   acaso: 33,3%")
POR_CAO = {}
for cao, g in d.groupby('cachorro'):
    k, n = int(g['acertos'].sum()), int(g['tentativas'].sum())
    b = jeffreys(k, n)
    POR_CAO[cao] = (k / n, *b.interval(0.95), len(g))
    print(f"  {cao:<8} {k} de {n} = {vir(k / n * 100)}%  IC 95% [{vir(b.ppf(0.025) * 100)}; {vir(b.ppf(0.975) * 100)}]  ({len(g)} pessoas)")

# CAMADA 2 · Pessoa a pessoa
print("\n" + "=" * 72)
print("  CAMADA 2 · PESSOA A PESSOA (36 pessoas, 20 tentativas cada)")
print("=" * 72)
d['p_acaso'] = stats.binom.sf(d['acertos'] - 1, N_TENT, ACASO)
MIN = int(d['acertos'].min())
PERFEITAS = int((d['acertos'] == N_TENT).sum())
print(f"  Pior sessão: {MIN} de 20 ({vir(MIN / 20 * 100, 0)}%). Sessões perfeitas: {PERFEITAS} de {len(d)}")
print(f"  Chance de acertar {MIN} ou mais de 20 escolhendo ao acaso: {vir(stats.binom.sf(MIN - 1, N_TENT, ACASO) * 100, 2)}%")
print(f"  Pessoas detectadas acima do acaso (p < 0,05): {int((d['p_acaso'] < 0.05).sum())} de {len(d)}")

# CAMADA 3 · Uma pessoa nova
print("\n" + "=" * 72)
print("  CAMADA 3 · UMA PESSOA NOVA (beta-binomial, Monte Carlo)")
print("=" * 72)
x = d['acertos'].values
def nll(p):
    a, b = np.exp(p)
    return -np.sum(special.gammaln(N_TENT + 1) - special.gammaln(x + 1) - special.gammaln(N_TENT - x + 1)
                   + special.betaln(x + a, N_TENT - x + b) - special.betaln(a, b))
AJ = optimize.minimize(nll, [np.log(20), np.log(1.5)])
ALFA, BETA_ = np.exp(AJ.x)
RHO = 1 / (ALFA + BETA_ + 1)
print(f"  Beta(α = {vir(ALFA, 2)}, β = {vir(BETA_, 2)}) · média {vir(ALFA / (ALFA + BETA_) * 100)}% · correlação intraclasse {vir(RHO, 3)}")
taxa_nova = rng.beta(ALFA, BETA_, SORTEIOS)
sessao_nova = rng.binomial(N_TENT, taxa_nova)
P_NOVA = {'acima de 90%': (taxa_nova > 0.9).mean(),
          'acima de 2 em 3': (taxa_nova > 2 / 3).mean(),
          'abaixo do acaso': (taxa_nova < ACASO).mean()}
for k_, v in P_NOVA.items():
    print(f"  P(acerto {k_} com uma pessoa nova): {vir(v * 100)}%")
print(f"  Numa sessão de 20 cheiradas com uma pessoa nova, acerto mediano: {int(np.median(sessao_nova))} de 20")

# CAMADA 4 · O tamanho do estresse importa?
print("\n" + "=" * 72)
print("  CAMADA 4 · O TAMANHO DO ESTRESSE IMPORTA?")
print("=" * 72)
R_VAS = stats.spearmanr(d['vas_dif'], d['acertos'])
R_FC = stats.spearmanr(fc['fc_mat'] - fc['fc_base'], fc['acertos'])
print(f"  Subida do estresse relatado × acertos: ρ = {vir(R_VAS.statistic, 2)} (p = {vir(R_VAS.pvalue, 2)}, N = {len(d)})")
print(f"  Subida da frequência cardíaca × acertos: ρ = {vir(R_FC.statistic, 2)} (p = {vir(R_FC.pvalue, 2)}, N = {len(fc)})")
print("  Com acerto perto do teto, não sobra variação para o tamanho do estresse explicar.")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · Os quatro cachorros contra o acaso
fig1, ax1 = plt.subplots(figsize=(11, 6))
caes = sorted(POR_CAO, key=lambda c: -POR_CAO[c][3])
y = np.arange(len(caes))[::-1]
for yi, c in zip(y, caes):
    p, lo_, hi_, n_ = POR_CAO[c]
    ax1.plot([lo_ * 100, hi_ * 100], [yi, yi], color=DOURADO, linewidth=4)
    ax1.plot(p * 100, yi, 'o', color=DOURADO, markersize=13)
    ax1.text(hi_ * 100 + 1, yi, f"{vir(p * 100)}%  ({n_} pessoas)", va='center', fontsize=11)
ax1.axvline(ACASO * 100, color=VERMELHO, linestyle='--', linewidth=1.8)
ax1.text(ACASO * 100 + 1, -0.45, 'acaso (1 em 3)', color=VERMELHO, fontsize=11)
ax1.set_ylim(-0.7, len(caes) - 0.5)
ax1.set_yticks(y)
ax1.set_yticklabels(caes, fontsize=12)
ax1.set_xlim(20, 108)
ax1.set_xlabel('Acerto ao achar a amostra estressada entre três potes (%) · IC 95%', fontsize=11)
ax1.set_title('Os quatro cachorros acharam o cheiro do estresse em mais de 9 de cada 10 tentativas\n'
              'posterior Beta de Jeffreys, 720 tentativas', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-155-grafico-01-os-quatro-cachorros.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · Pessoa a pessoa
fig2, ax2 = plt.subplots(figsize=(11, 6))
cont = d['acertos'].value_counts().reindex(range(0, 21), fill_value=0)
ax2.bar(cont.index, cont.values, color=DOURADO, width=0.8, label='pessoas no estudo')
acaso = stats.binom.pmf(range(0, 21), N_TENT, ACASO) * len(d)
ax2.plot(range(0, 21), acaso, 'o-', color=VERMELHO, linewidth=2, label='o que o acaso daria')
ax2.set_xticks(range(0, 21, 2))
ax2.set_xlabel('Acertos em 20 cheiradas, por pessoa', fontsize=12)
ax2.set_ylabel('Número de pessoas', fontsize=12)
ax2.legend(frameon=False, fontsize=11)
ax2.set_title(f'Nenhuma das 36 pessoas passou despercebida: a pior sessão teve {MIN} acertos em 20\n'
              f'{PERFEITAS} pessoas tiveram o estresse achado nas 20 tentativas', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-155-grafico-02-pessoa-a-pessoa.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Uma pessoa nova
fig3, ax3 = plt.subplots(figsize=(11, 6))
xs = np.linspace(0.3, 0.998, 700)
dens = stats.beta(ALFA, BETA_).pdf(xs)
ax3.plot(xs * 100, dens, color=VERDE, linewidth=3)
ax3.fill_between(xs * 100, dens, where=xs > 0.9, color=VERDE, alpha=0.25)
ax3.axvline(ACASO * 100, color=VERMELHO, linestyle='--', linewidth=1.8)
ax3.text(ACASO * 100 + 1, dens.max() * 0.85, 'acaso', color=VERMELHO, fontsize=11)
ax3.annotate(f"acerto acima de 90%\nem {vir(P_NOVA['acima de 90%'] * 100, 0)} de cada 100 pessoas novas",
             xy=(94, dens.max() * 0.3), xytext=(62, dens.max() * 0.6), fontsize=12, color=VERDE,
             arrowprops=dict(arrowstyle='->', color=VERDE))
ax3.set_xlabel('Acerto do cachorro com uma pessoa que ele nunca cheirou (%)', fontsize=12)
ax3.set_ylabel('Densidade', fontsize=12)
ax3.set_title('Com uma pessoa nova, o acerto quase nunca fica perto do acaso\n'
              'beta-binomial ajustado às 36 sessões · ' + f'{SORTEIOS:,}'.replace(",", ".") + ' pessoas simuladas',
              fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-155-grafico-03-uma-pessoa-nova.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · O tamanho do estresse
fig4, ax4 = plt.subplots(figsize=(11, 6))
jit = rng.uniform(-0.15, 0.15, len(d))
ax4.scatter(d['vas_dif'], d['acertos'] + jit, s=70, color=DOURADO, edgecolor='#333', alpha=0.85)
ax4.axhline(N_TENT * ACASO, color=VERMELHO, linestyle='--', linewidth=1.5)
ax4.text(d['vas_dif'].max(), N_TENT * ACASO + 0.4, 'acaso', color=VERMELHO, ha='right', fontsize=11)
ax4.set_ylim(0, 21)
ax4.set_xlabel('Quanto o estresse relatado subiu (escala de 0 a 10)', fontsize=12)
ax4.set_ylabel('Acertos em 20 cheiradas', fontsize=12)
ax4.set_title('Pouco ou muito estresse, o cachorro acha do mesmo jeito\n'
              f'correlação de Spearman ρ = {vir(R_VAS.statistic, 2)}, 36 pessoas', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-155-grafico-04-o-tamanho-do-estresse.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Com só o cheiro do hálito e do suor, quatro cachorros acharam a amostra da pessoa estressada
em 94 de cada 100 tentativas, quando o acaso daria 33. Nenhuma das 36 pessoas passou
despercebida. Na pior sessão, o cachorro acertou 14 de 20.**

O estresse tem cheiro, e o cachorro sente esse cheiro sem precisar ver a pessoa, ouvir a voz
ou perceber um gesto. E não é só perceber. Com o cheiro de medo ou de estresse, cachorros comuns
ficam mais agitados e mais cautelosos, mesmo quando a pessoa é desconhecida.

Ao longo de meses, a ligação continua. Em 58 duplas, as donas com mais estresse acumulado
tinham cachorros com mais estresse acumulado. O estudo não diz quem passa para quem, mas diz
que os dois andam juntos.

*Quando foi a última vez que o seu cachorro percebeu antes de você dizer qualquer coisa?*

---

### ⚠️ Limitações do Modelo

- O estudo central tem só 4 cachorros, todos treinados para a tarefa, e 2 deles com poucas
  sessões. O resultado mostra que o cheiro existe e pode ser achado, não que todo cachorro em
  casa faça o mesmo.
- O teste compara a mesma pessoa calma e estressada, poucos minutos depois. Na vida real, o
  cheiro do estresse se mistura com perfume, comida e o resto do dia.
- O estresse foi provocado por uma tarefa de contas de cabeça sob pressão, de curta duração.
  Estresse longo ou de outro tipo pode ter outro cheiro.
- O beta-binomial foi ajustado com os 4 cachorros juntos, porque há poucas sessões por
  cachorro. A pessoa nova simulada é uma leitura do modelo, não uma pessoa medida.
- Com quase todas as sessões acima de 17 acertos, a correlação com o tamanho do estresse não
  tem espaço para aparecer. A ausência de relação não prova que o tamanho não importa.
- A planilha traz frequência cardíaca de 26 pessoas (de 87 para 101 batimentos por minuto), e o
  texto do artigo fala em 25 (de 91 para 105). O notebook usa a planilha.
- D'Aniello et al. 2018 e Parr-Cortes et al. 2024 entram pelos resumos, sem números no modelo.
- Sundman et al. 2019 é associação entre o cortisol do dono e o do cachorro, só com donas
  mulheres e duas raças de pastoreio. A correção publicada em 2020 não foi lida.
- Sem ×0,80, porque são acertos contados num teste duplo-cego e medidas de cortisol.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
